# 🍁 LeafID: Colab runner

**Every session:** Runtime → Change runtime type → **T4 GPU**, then run sections **1–3** (about 5 min).
After that, run only the cells you own.

**Got disconnected?** Run sections 1–3 again, then re-run the same training cell. Every run uses `--resume`,
so it continues from the last finished epoch saved on your Google Drive.

| Section | What | When |
|---|---|---|
| 1. Setup | clone/update repo, install | every session |
| 2. Drive | where checkpoints + reports are saved | every session |
| 3. Data | download LeafSnap, build the fast image cache | every session |
| 4. Smoke test | 2-epoch run that checks the whole pipeline | once, or after code changes |
| 5. Experiments | the real training runs | one cell per run, whoever owns it |
| 6. Compare | evaluate on test + results table + curves | after runs finish |
| 7. App | launch the Gradio demo | demo / screenshots / video |

## 1. Setup

In [ ]:
import os, sys

BRANCH = "main"  # @param {type:"string"}
REPO_URL = "https://github.com/akshayanbalathas/leafid.git"

!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: Runtime -> Change runtime type -> T4 GPU"
if not os.path.exists("/content/leafid"):
    !git clone -q {REPO_URL} /content/leafid
%cd /content/leafid
!git fetch -q origin && git checkout -q {BRANCH} && git pull -q --ff-only origin {BRANCH}
!pip install -q -r requirements.txt && pip install -q -e .
sys.path.insert(0, "/content/leafid/src")  # so this notebook can import leafid without a restart
!git log --oneline -1

## 2. Google Drive
Checkpoints and reports are written straight to Drive **during** training, so a disconnect loses at most one epoch.
Untick `USE_DRIVE` only for throwaway experiments.

In [ ]:
USE_DRIVE = True  # @param {type:"boolean"}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT = "/content/drive/MyDrive/leafid"
else:
    OUTPUT = "/content/leafid_output"  # deleted when the session ends
os.makedirs(OUTPUT, exist_ok=True)
os.environ["LEAFID_OUTPUT"] = OUTPUT  # picked up by every !python -m leafid... command below
print("checkpoints + reports ->", OUTPUT)

## 3. Data
Downloads LeafSnap (~880 MB, public, no Kaggle login needed), then shrinks every image once to 256 px into
`/content/cache_256`. Training then skips decoding 800×600 JPEGs every epoch, which is the bottleneck on Colab's 2 CPUs.
The cache lives on the local disk (fast) and is rebuilt each session in a few minutes. **Don't** put it on Drive.

In [ ]:
import kagglehub

try:  # optional: only needed if Kaggle ever refuses anonymous downloads (add in the 🔑 Secrets panel)
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    pass

os.environ["LEAFID_DATA"] = kagglehub.dataset_download("xhlulu/leafsnap-dataset")
os.environ["LEAFID_CACHE"] = "/content/cache_256"
assert os.path.exists("splits/splits.csv"), "splits/splits.csv missing: run section 1 again to pull main"
print("dataset:", os.environ["LEAFID_DATA"])

!python -m leafid.cache

## 4. Smoke test (~10 min)
Checks data loading, both training stages, checkpointing and evaluation before anyone spends hours training.
Saves to `/content/smoke`, not Drive. Note the **seconds per epoch**: it tells you how many epochs fit in a session.
Field val top-1 just needs to be clearly above chance (1/185 ≈ 0.5%).

In [ ]:
from leafid.data import image_root, make_loader

x, y = next(iter(make_loader("train", train=True, batch_size=8)))
print("images from:", image_root())
print("batch:", tuple(x.shape), x.dtype, "labels:", y.tolist())  # expect (8, 3, 224, 224) torch.float32

!python -m leafid.train --model resnet50 --epochs 2 --freeze-epochs 1 --run smoke --save-dir /content/smoke
!python -m leafid.evaluate --checkpoint /content/smoke/checkpoints/smoke.pt --save-dir /content/smoke

## 5. Experiments
One cell per run. **Run only the ones you own**, so we don't duplicate GPU time across accounts.
Every cell ends in `--resume`: after a disconnect, run sections 1–3 and then the same cell again.

| Run | What it tests | Owner |
|---|---|---|
| `simple_cnn` | small CNN from scratch (baseline) | Akshayan |
| `enhanced_cnn` | deeper CNN (2 convs per block, 512 features), from scratch | TODO |
| `res_cnn` | + residual connections, still from scratch | TODO |
| `resnet50_probe` | pretrained ImageNet features, only the head trains | TODO |
| `resnet50_full` | two-stage fine-tune (**the model the app uses**) | TODO |
| `resnet50_labonly` | trained on lab photos only → lab-to-field domain shift | TODO |

In [ ]:
!python -m leafid.train --model simple_cnn --epochs 30 --lr 1e-3 --run simple_cnn --resume

In [ ]:
!python -m leafid.train --model enhanced_cnn --epochs 30 --lr 1e-3 --run enhanced_cnn --resume

In [ ]:
!python -m leafid.train --model res_cnn --epochs 30 --lr 1e-3 --run res_cnn --resume

In [ ]:
!python -m leafid.train --model resnet50 --epochs 10 --freeze-epochs 10 --run resnet50_probe --resume

In [ ]:
!python -m leafid.train --model resnet50 --epochs 15 --freeze-epochs 3 --run resnet50_full --resume

In [ ]:
!python -m leafid.train --model resnet50 --epochs 15 --freeze-epochs 3 --train-sources lab --run resnet50_labonly --resume

## 6. Evaluate + compare
Evaluates every **finished** run on your Drive against the test split, then shows the results table and
validation curves. Teammates' runs live on their own Drives: share the `reports/` folders to combine results.

In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt

OUT = os.environ["LEAFID_OUTPUT"]
for ckpt in sorted(glob.glob(f"{OUT}/checkpoints/*.pt")):
    run = os.path.basename(ckpt)[:-3]
    if run.endswith("_last") or os.path.exists(f"{OUT}/checkpoints/{run}_last.pt"):
        print(f"skipping {run}: still in progress")
        continue
    !python -m leafid.evaluate --checkpoint "{ckpt}"

rows = []
for p in sorted(glob.glob(f"{OUT}/reports/*/metrics.json")):
    m = json.load(open(p))
    rows.append({"run": p.split("/")[-2], "model": m["model"], "lab_top1": m["lab"]["top1"],
                 "field_top1": m["field"]["top1"], "field_top5": m["field"]["top5"]})
display(pd.DataFrame(rows).round(3))

plt.figure(figsize=(7, 4))
for p in sorted(glob.glob(f"{OUT}/reports/*/history.csv")):
    h = pd.read_csv(p)
    plt.plot(h["epoch"], h["val_field_top1"], marker="o", label=p.split("/")[-2])
plt.xlabel("epoch"); plt.ylabel("val field top-1"); plt.title("Validation accuracy on field photos")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 6b. Grad-CAM check of a confused pair (~2 min, no training)
`resnet50_full` predicts **every** `pinus_virginiana` (Virginia pine) test photo as `prunus_virginiana` (chokecherry).
A pine has needles and a chokecherry has broad leaves, so a 93% model shouldn't mix them up. This cell shows the test photos,
where the model looked (Grad-CAM), and training photos of both species. If the heatmap sits on the background, the model
learned a shortcut. If the test photos look like the chokecherry training photos, the dataset labels are wrong.
Also try `TRUE = quercus_muehlenbergii`, `PRED = prunus_sargentii` (confused in both directions).

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from leafid.data import image_root, build_transforms, display_transform
from leafid.models import load_checkpoint, gradcam_layer
from leafid.gradcam import GradCAM, overlay

RUN = "resnet50_full"  # @param {type:"string"}
TRUE = "pinus_virginiana"  # @param {type:"string"}
PRED = "prunus_virginiana"  # @param {type:"string"}
N = 6  # images per row

device = "cuda" if torch.cuda.is_available() else "cpu"
model, name, classes = load_checkpoint(f"{os.environ['LEAFID_OUTPUT']}/checkpoints/{RUN}.pt", device)
gradcam = GradCAM(model, gradcam_layer(model, name))
root, show, to_tensor = image_root(), display_transform(), build_transforms(train=False)
splits = pd.read_csv("splits/splits.csv")

def pick(species, split, source="field"):
    df = splits[(splits.species == species) & (splits.split == split) & (splits.source == source)]
    return df.sample(min(N, len(df)), random_state=0)

print("images per split/source:")
display(splits[splits.species.isin([TRUE, PRED])].groupby(["species", "source", "split"]).size().unstack(fill_value=0))

rows = [(pick(TRUE, "test"), f"TEST field, labelled {TRUE}", False),
        (pick(TRUE, "test"), "same photos: Grad-CAM for the predicted class", True),
        (pick(TRUE, "train"), f"TRAIN field, labelled {TRUE}", False),
        (pick(PRED, "train"), f"TRAIN field, labelled {PRED}", False)]
fig, axes = plt.subplots(len(rows), N, figsize=(2.6 * N, 3.4 * len(rows)), squeeze=False)
for r, (df, title, with_cam) in enumerate(rows):
    for c, ax in enumerate(axes[r]):
        ax.axis("off")
        if c >= len(df):
            continue
        img = Image.open(root / df.iloc[c].image_path).convert("RGB")
        if r < 2:  # test photos: show what the model predicts and where it looked
            cam, logits = gradcam(to_tensor(img).unsqueeze(0).to(device))
            p = torch.softmax(logits.float(), 1)[0]
            k = int(p.argmax())
            ax.set_title(f"{classes[k]}\n{p[k]:.0%}", fontsize=8,
                         color="green" if classes[k] == TRUE else "red")
            ax.imshow(overlay(show(img), cam) if with_cam else show(img))
        else:
            ax.imshow(show(img))
    axes[r][0].text(0, 1.3 if r < 2 else 1.04, title, transform=axes[r][0].transAxes,
                    fontsize=10, weight="bold", va="bottom")
plt.tight_layout(); plt.show()

## 7. Launch the app
Opens a public `gradio.live` link. The first launch downloads CLIP and the LLM (a few minutes);
untick `USE_LLM` for a fast start. Stop the cell (■) when done: a running app keeps using your GPU quota.

In [ ]:
RUN = "resnet50_full"  # @param {type:"string"}
USE_LLM = True  # @param {type:"boolean"}

CKPT = f"{os.environ['LEAFID_OUTPUT']}/checkpoints/{RUN}.pt"
assert os.path.exists(CKPT), f"{CKPT} not found: train it first (section 5) or pick another RUN"
FLAGS = "" if USE_LLM else "--no-llm"
!python app/app.py --checkpoint "{CKPT}" --share {FLAGS}

## Getting results into the repo
Checkpoints stay on Drive (too big for GitHub). The **reports** are small: download
`MyDrive/leafid/reports/<run>/` (history.csv, metrics.json, confusion_field.png, confusions.csv),
copy them into the repo's `reports/` folder locally, and open a PR. The README results table is filled from `metrics.json`.